In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator

In [3]:
def random_sample_imp(df ,col, random_state=9009):
    np.random.seed(random_state)

    null_count = df[col].isnull().sum()

    if null_count > 0:
        random_value = df[col].dropna().sample(
            n=null_count,
            replace=True,
            random_state=random_state
        ).values

        df_ = df.copy()
        df_.loc[df_[col].isnull(), col] = random_value

        return df_

    return df


#### Q1 - Missing Indicator + Random Sample Imputation

In [4]:
# df with missing values

df = pd.DataFrame({
    'Age': [20, np.nan, np.nan, np.nan, 21, 38, 23, 25, 19, np.nan, np.nan],
    'Salary': [np.nan, 45000, np.nan, 50000, 34000, np.nan, 30000, 25000, 28000,np.nan, 39000 ],
})
df

,Age,Salary
0,20.0,NaN
1,NaN,45000.0
2,NaN,NaN
3,NaN,50000.0
4,21.0,34000.0
5,38.0,NaN
6,23.0,30000.0
7,25.0,25000.0
8,19.0,28000.0
9,NaN,NaN


In [5]:
# Adding missing col

indicator = MissingIndicator(features='missing-only')
indicator_array = indicator.fit_transform(df)

indicator_col = [df.columns[col] + '_Missing' for col in indicator.features_]
indicator_df = pd.DataFrame(indicator_array.astype(int), columns=indicator_col)
indicator_df

,Age_Missing,Salary_Missing
0,0,1
1,1,0
2,1,1
3,1,0
4,0,0
5,0,1
6,0,0
7,0,0
8,0,0
9,1,1


In [6]:
# Random Sample Imputation

imputed_df = df.copy()

for col in df.columns:
    imputed_df = random_sample_imp(imputed_df,col)

imputed_df

,Age,Salary
0,20.0,25000.0
1,25.0,45000.0
2,38.0,34000.0
3,21.0,50000.0
4,21.0,34000.0
5,38.0,50000.0
6,23.0,30000.0
7,25.0,25000.0
8,19.0,28000.0
9,25.0,25000.0


In [7]:
# Original vs Imputed

imputed_df = pd.concat([imputed_df,indicator_df], axis=1)
imputed_df

print(f'Orignial DataFrame: \n{df}')
print(f'Imputed DataFrame: \n{imputed_df}')

Orignial DataFrame: 
     Age   Salary
0   20.0      NaN
1    NaN  45000.0
2    NaN      NaN
3    NaN  50000.0
4   21.0  34000.0
5   38.0      NaN
6   23.0  30000.0
7   25.0  25000.0
8   19.0  28000.0
9    NaN      NaN
10   NaN  39000.0
Imputed DataFrame: 
     Age   Salary  Age_Missing  Salary_Missing
0   20.0  25000.0            0               1
1   25.0  45000.0            1               0
2   38.0  34000.0            1               1
3   21.0  50000.0            1               0
4   21.0  34000.0            0               0
5   38.0  50000.0            0               1
6   23.0  30000.0            0               0
7   25.0  25000.0            0               0
8   19.0  28000.0            0               0
9   25.0  25000.0            1               1
10  21.0  39000.0            1               0


#### Q2 - KNN Imputer (Multivariate Imputation)

In [8]:
# Create df

df = pd.DataFrame({
    'Age': [21, np.nan, np.nan, np.nan, 21, 38, 23, 25, 19, np.nan, np.nan],
    'Days': [15, 10, 18, 24, np.nan, np.nan, 21, np.nan, 30, 21, 11 ],
    'Income': [5000, 8900, np.nan, 7300, np.nan, 3700, 6980, np.nan, np.nan, np.nan, 10000],
    'Rating': [4.5, 3.8, 4.9, 4.1, np.nan, np.nan, 3.0, 3.9, 4.6, np.nan, 2.0],

})
df

,Age,Days,Income,Rating
0,21.0,15.0,5000.0,4.5
1,NaN,10.0,8900.0,3.8
2,NaN,18.0,NaN,4.9
3,NaN,24.0,7300.0,4.1
4,21.0,NaN,NaN,NaN
5,38.0,NaN,3700.0,NaN
6,23.0,21.0,6980.0,3.0
7,25.0,NaN,NaN,3.9
8,19.0,30.0,NaN,4.6
9,NaN,21.0,NaN,NaN


In [9]:
# KNN

knn = KNNImputer(n_neighbors=3, weights='distance')

imputed_df = pd.DataFrame(knn.fit_transform(df), columns=df.columns)
imputed_df

,Age,Days,Income,Rating
0,21.000000,15.000000,5000.000000,4.500000
1,24.957730,10.000000,8900.000000,3.800000
2,23.571133,18.000000,6206.287682,4.900000
3,24.728994,24.000000,7300.000000,4.100000
4,21.000000,15.000000,5000.000000,4.500000
5,38.000000,29.627023,3700.000000,4.187003
6,23.000000,21.000000,6980.000000,3.000000
7,25.000000,14.874999,8309.513215,3.900000
8,19.000000,30.000000,6690.370482,4.600000
9,23.000000,21.000000,6980.000000,3.000000


In [13]:
# Original vs Imputed

print(f'Orignial DataFrame: \n{df}\n')
print(f'Imputed DataFrame: \n{imputed_df.round(1)}\n')

Orignial DataFrame: 
     Age  Days   Income  Rating
0   21.0  15.0   5000.0     4.5
1    NaN  10.0   8900.0     3.8
2    NaN  18.0      NaN     4.9
3    NaN  24.0   7300.0     4.1
4   21.0   NaN      NaN     NaN
5   38.0   NaN   3700.0     NaN
6   23.0  21.0   6980.0     3.0
7   25.0   NaN      NaN     3.9
8   19.0  30.0      NaN     4.6
9    NaN  21.0      NaN     NaN
10   NaN  11.0  10000.0     2.0

Imputed DataFrame: 
     Age  Days   Income  Rating
0   21.0  15.0   5000.0     4.5
1   25.0  10.0   8900.0     3.8
2   23.6  18.0   6206.3     4.9
3   24.7  24.0   7300.0     4.1
4   21.0  15.0   5000.0     4.5
5   38.0  29.6   3700.0     4.2
6   23.0  21.0   6980.0     3.0
7   25.0  14.9   8309.5     3.9
8   19.0  30.0   6690.4     4.6
9   23.0  21.0   6980.0     3.0
10  24.3  11.0  10000.0     2.0



In [16]:
# Experiment knn n_neighbours=2

knn = KNNImputer(n_neighbors=2, weights='distance')

imputed_df_2 = pd.DataFrame(knn.fit_transform(df), columns=df.columns)
imputed_df_2

,Age,Days,Income,Rating
0,21.000000,15.000000,5000.000000,4.500000
1,24.957905,10.000000,8900.000000,3.800000
2,23.726153,18.000000,5911.056085,4.900000
3,24.730782,24.000000,7300.000000,4.100000
4,21.000000,15.000000,5000.000000,4.500000
5,38.000000,29.696265,3700.000000,4.184375
6,23.000000,21.000000,6980.000000,3.000000
7,25.000000,14.666665,8366.666832,3.900000
8,19.000000,30.000000,7164.011505,4.600000
9,23.000000,21.000000,6980.000000,3.000000


In [17]:
print(f'N_neighbors = 3: \n{imputed_df.round(1)}\n')
print(f'N_neighbors = 2: \n{imputed_df_2.round(1)}\n')

N_neighbors = 3: 
     Age  Days   Income  Rating
0   21.0  15.0   5000.0     4.5
1   25.0  10.0   8900.0     3.8
2   23.7  18.0   5911.1     4.9
3   24.7  24.0   7300.0     4.1
4   21.0  15.0   5000.0     4.5
5   38.0  29.7   3700.0     4.2
6   23.0  21.0   6980.0     3.0
7   25.0  14.7   8366.7     3.9
8   19.0  30.0   7164.0     4.6
9   23.0  21.0   6980.0     3.0
10  24.3  11.0  10000.0     2.0

N_neighbors = 2: 
     Age  Days   Income  Rating
0   21.0  15.0   5000.0     4.5
1   25.0  10.0   8900.0     3.8
2   23.7  18.0   5911.1     4.9
3   24.7  24.0   7300.0     4.1
4   21.0  15.0   5000.0     4.5
5   38.0  29.7   3700.0     4.2
6   23.0  21.0   6980.0     3.0
7   25.0  14.7   8366.7     3.9
8   19.0  30.0   7164.0     4.6
9   23.0  21.0   6980.0     3.0
10  24.3  11.0  10000.0     2.0

